In [1]:
from openai import OpenAI
import os

openai_client = OpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1"
)

In [2]:
from ingest import load_faq_data
documents = load_faq_data()

In [3]:
documents[10]

{'id': '316180784f',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: How many hours per week am I expected to spend on this course?',
 'answer': 'It depends on your background and previous experience with modules. It is expected to require about 5 - 15 hours per week.\n\nYou can also calculate it yourself using [this data](https://github.com/DataTalksClub/zoomcamp-analytics/tree/main/data/de-zoomcamp-2023) and then update this answer.'}

In [4]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

113

In [5]:
documents = documents_llm

In [6]:
doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [7]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [8]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [9]:
import json
user_prompt = json.dumps(doc)

In [10]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [11]:
response = openai_client.responses.parse(
    model="openai/gpt-5.4-mini",
    input=messages,
    text_format=Questions,
    max_output_tokens=2000,
    reasoning={
        "effort": "low"
    }
)

In [12]:
result = response.output_parsed

print(result)

questions=['Can I join the course late if I only found it now?', 'Is it still okay to start the course after it has already begun?', 'If I’m joining the course late, what do I need to do to still get a certificate?', 'Do I miss out on certification if I submit my project after submissions close?', 'Can I still enroll now, or is it too late to take part in the course?']


In [13]:
response.output_parsed.questions

['Can I join the course late if I only found it now?',
 'Is it still okay to start the course after it has already begun?',
 'If I’m joining the course late, what do I need to do to still get a certificate?',
 'Do I miss out on certification if I submit my project after submissions close?',
 'Can I still enroll now, or is it too late to take part in the course?']

In [ ]:
from evaluation_utils import llm_structured

In [15]:
result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

['I just found this course — is it too late to join now?', 'Can I still sign up even if I discovered the course late?', 'Am I allowed to start the course after it has already begun?', 'If I join now, will I still be able to get a certificate?', 'What do I need to do to earn the certificate if I’m joining late?']


In [16]:
usage.input_tokens, usage.output_tokens

(207, 87)

In [ ]:
from evaluation_utils import calc_price

In [18]:
cost = calc_price(usage)

cost

{'input_cost': 0.00015525,
 'output_cost': 0.00039150000000000003,
 'total_cost': 0.00054675}

In [19]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': 'I just found this course — is it too late to join now?',
  'document': '74eb249bbf'},
 {'question': 'Can I still sign up even if I discovered the course late?',
  'document': '74eb249bbf'},
 {'question': 'Am I allowed to start the course after it has already begun?',
  'document': '74eb249bbf'},
 {'question': 'If I join now, will I still be able to get a certificate?',
  'document': '74eb249bbf'},
 {'question': 'What do I need to do to earn the certificate if I’m joining late?',
  'document': '74eb249bbf'}]

In [ ]:
from evaluation_utils import llm_structured_retry

In [21]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [22]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/5 [00:00<?, ?it/s]

In [ ]:
for m in openai_client.models.list().data:
    print(m.id)

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from spam import map_progress

In [24]:
with ThreadPoolExecutor(max_workers=1) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

  0%|          | 0/113 [00:00<?, ?it/s]

In [25]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

565

In [ ]:
from spam import calc_price

total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

total_cost

0.08653200000000004

In [ ]:
from spam import calc_total_price

calc_total_price(usages)

0.08653200000000004

In [28]:
import pandas as pd

df_ground_truth = pd.DataFrame(ground_truth)

In [30]:
df_ground_truth.to_csv("data/ground_truth-new.csv", index=False)

In [31]:
len(df_ground_truth)

565